# Practical 2: Time Series Analysis & Forecasting
## Dataset: CO2 Concentration.xls
## Objective: Handle multi-column dates, perform additive decomposition, exponential smoothing, and stationarity testing

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
df = pd.read_csv("CO2 Concentration.xls")

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
df['year_month'] = df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2)
df.head(5)

In [ ]:
#convert to datetime format
df['year_month'] = pd.to_datetime(df['year_month'])
df = df.set_index('year_month')

In [ ]:
df.head(5)

In [ ]:
df.drop(columns = ['Year','Month'], inplace = True)

In [ ]:
df.head(5)

In [ ]:
sns.lineplot(df)

### 📊 How to Read This Graph:
- CO2 concentration rises steadily from ~315 ppm to 390+ ppm → **strong upward trend**.
- Seasonal peaks and dips repeat every year with **constant height** → **Additive seasonality**.
- Unlike AirPassengers where peaks grew taller, here the wave height stays the same throughout.

In [ ]:
#the model is additive model, because the pattern is same throughout

In [ ]:
#decomposition of the time series - additive model
result = seasonal_decompose(df[['CO2 Concentration']], model = 'additive', period = 12)
result.plot()
plt.show()

### 📊 How to Read This Decomposition:
- **Observed**: Raw CO2 data.
- **Trend**: Smooth upward curve showing atmospheric CO2 accumulation over decades.
- **Seasonal**: Uniform annual wave pattern (constant height). Peak = May, Dip = October.
- **Resid**: Small random noise — very clean decomposition.

In [ ]:
import pymannkendall as mk

In [ ]:
#Perform the Mann-Kendall test
#H0: There is no monotonic trend in the series
mk.original_test(df['CO2 Concentration'])

In [ ]:
#Train test split
#Train test splitting
train_df = df[:int(df.shape[0]*0.7)]
test_df = df[int(df.shape[0]*0.7):]

In [ ]:
train_df.head(5)

In [ ]:
#single exponential smoothing model
from statsmodels.tsa.api import SimpleExpSmoothing
model = SimpleExpSmoothing(train_df)
model_single_fit = model.fit()

In [ ]:
forecast_single = model_single_fit.forecast(len(test_df))
print(forecast_single)

In [ ]:
model_single_fit.params

In [ ]:
plt.plot(df, label ="Original Data")
plt.plot(model_single_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_single, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("CO2 Concentration")
plt.title("Single exponential smoothing")
plt.legend()
plt.show()

In [ ]:
#DOUBLE EXPONENETIAL SMOOTHING (Holt's model)
from statsmodels.tsa.api import Holt

In [ ]:
model_double = Holt(train_df)
model_double_fit = model_double.fit()

In [ ]:
forecast_double = model_double_fit.forecast(len(test_df))
print(forecast_double)

In [ ]:
model_double_fit.params

In [ ]:
plt.plot(df, label ="Original Data")
plt.plot(model_double_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_double, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("CO2 concentration")
plt.title("Double exponential smoothing(Holt's model)")
plt.legend()
plt.show()

In [ ]:
#TRIPLE EXPONENTIAL SMOOTHING (HOLT_WINTER'S MODEL)
from statsmodels.tsa.api import ExponentialSmoothing

In [ ]:
model_triple = ExponentialSmoothing(train_df, seasonal_periods = 12, trend = "add", seasonal = "add")
#from the pegel's chart, since it looks similar to the additive-additive model,
#therefore both trend and seasonality have been taken as additive
model_triple_fit = model_triple.fit()

In [ ]:
model_triple_fit.params

In [ ]:
forecast_triple = model_triple_fit.forecast(len(test_df))
print(forecast_triple)

In [ ]:
plt.plot(df, label ="Original Data")
plt.plot(model_triple_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_triple, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("CO2 Concentration")
plt.title("Triple exponential smoothing (Holt-winter's model)")
plt.legend()
plt.show()

In [ ]:
#using add-multi model
model_triple_mul = ExponentialSmoothing(train_df, seasonal_periods = 12, trend = "add", seasonal = "mul")

model_triple_fit_mul = model_triple_mul.fit()

In [ ]:
forecast_triple_mul = model_triple_fit_mul.forecast(len(test_df))
print(forecast_triple_mul)

In [ ]:
plt.plot(df, label ="Original Data")
plt.plot(model_triple_fit_mul.fittedvalues, label= "Fitted values")
plt.plot(forecast_triple_mul, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("CO2 Concentration")
plt.title("Triple exponential smoothing (Holt-winter's model)")
plt.legend()
plt.show()

In [ ]:
#testing for the accuracy of the two models
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

In [ ]:
mape_test_add = mean_absolute_percentage_error(test_df['CO2 Concentration'], forecast_triple)
print("MAPE Test for Test Data:",mape_test_add)

In [ ]:
mape_test_mul = mean_absolute_percentage_error(test_df['CO2 Concentration'], forecast_triple_mul)
print("MAPE Test for Test Data:",mape_test_mul)

In [ ]:
#No difference in both methodologies

### 📝 Why are Additive and Multiplicative MAPE scores identical?
- Because CO2 seasonal oscillations have **constant height** throughout.
- When seasonal amplitude does NOT expand, both additive and multiplicative formulas produce the same forecast.

In [ ]:
#ADF Test
#H0: Series is not stationary, ie,e series has a unit root
#H1: Series is stationary, ie,e series has no unit root

In [ ]:
from statsmodels.tsa.stattools import adfuller 
result = adfuller(df['CO2 Concentration'])

print("ADF Statistic:", result[0])
print("p-value:",result[1])

In [ ]:
#p-value > 0.05 : fail to reject to H0, i.e, series is not stationary

In [ ]:
#KPSS Test
#H0: Series is trend stationary, ie,e series has no unit root
#H1: Series is non-stationary, ie,e series has a unit root

##It is very specific to trend (KPSS)

In [ ]:
from statsmodels.tsa.stattools import kpss
kp = kpss(df['CO2 Concentration'])
p = kp[1]

print("p-value for KPSS Test(untransformed) = ",p)